## For DNAase controls of 80K NGN2 derived neurons
- control groups covered: 'GC_DNase_positive', 'GC_DNase_negative_brain', 'GC_DNase_negative_blood', 'GC_DNase_positive_shuffeled', 'GC_DNase_negative_brain_shuffeled', 'GC_DNase_negative_blood_shuffeled'
- lift over to hg38 
- ending of the lines is important

In [1]:
from importlib import reload
import pandas as pd
import sys
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [2]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


shuffled_group = ['GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled']

dnase_control_groups = ['GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood'] + shuffled_group


def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    Case: GC_DNase_positive_shuffeled: (all have ":chr" pattern)
            header: GC_DNase_positive_shuffeled:chr1:121484605-121484874_active_count_114
    """

    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    name = row[col_name]

    if name.startswith('GC_DNase_positive:') or name.startswith('GC_DNase_negative_brain:') or name.startswith('GC_DNase_negative_blood:') or ('_shuffeled:' in name):
        row[col_ref] = 'GRCh38'
        row[col_category] = 'scrambled'
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
    return row


In [3]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
pre_metadata_df

# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(dnase_control_groups)].copy()
pre_metadata_df_filtered # 971

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'NA'
pre_metadata_df_filtered[col_class] = 'NA'
pre_metadata_df_filtered[col_source] = 'NA'
# all are GRCh37
pre_metadata_df_filtered[col_ref] = 'GRCh38' # GRCh37
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])

In [4]:
pre_metadata_df_filtered.head()

# write to bed file (hg19)
# pre_metadata_df[[col_chr, col_start, col_end, col_name,  ]]

,name,sequence,tmp_label,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
75480,GC_DNase_positive:chr1:121484605-121484874_act...,GGTTGTTACAGAACTAGGCTCATTGATAATAACAGTGGTGATTAGA...,GC_DNase_positive,scrambled,element inactive control,NA,GRCh38,chr1,121484605,121484874,.,NA,NA,NA,NA,
75481,GC_DNase_positive:chr1:153959246-153959515_act...,TTGGGCTCCTCGGGTGTATTTAAAAAAATGTTTTGGCAGCTCAGTG...,GC_DNase_positive,scrambled,element inactive control,NA,GRCh38,chr1,153959246,153959515,.,NA,NA,NA,NA,
75482,GC_DNase_positive:chr1:156186296-156186565_act...,ATTATGTGGTCTCCACCAAATAGCCACTTTTCCATTTTTACCAGAG...,GC_DNase_positive,scrambled,element inactive control,NA,GRCh38,chr1,156186296,156186565,.,NA,NA,NA,NA,
75483,GC_DNase_positive:chr1:161582223-161582492_act...,AGAGAGACAGGGTCTTTGGCCAAAATGGAGGGCTCTTTTCTAACCA...,GC_DNase_positive,scrambled,element inactive control,NA,GRCh38,chr1,161582223,161582492,.,NA,NA,NA,NA,
75484,GC_DNase_positive:chr1:223254657-223254926_act...,ACAAAAAGATCAGTGTTTCTGGATTTGATATAGGAAAAGAAAAAAG...,GC_DNase_positive,scrambled,element inactive control,NA,GRCh38,chr1,223254657,223254926,.,NA,NA,NA,NA,


In [5]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

# # Write DataFrame to TSV file
# pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
# import os
# os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/dnase_controls/dnase_controls.metadata.tsv.gz')

### Lift over
- with bed file 
    1. add coordinates using bed files (remove label from name)
    2. for shuffled: use the header to regions + set to scrambled
    3. write bed file of hg19
    4. liftover => bed file with hg38
    5. join and rename columns (old_chr, old_start, old_end) and chr, start, end

In [6]:
dnase_control_coordinates = {
'GC_DNase_positive': '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_PosControls.bed.gz',
'GC_DNase_negative_brain': '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.Brain.bed.gz',
'GC_DNase_negative_blood': '/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.BloodT-cell.bed.gz',
}

In [7]:
# remove label from name
pre_metadata_df_filtered['name_no_label'] = pre_metadata_df_filtered['name'].apply(lambda name: ':'.join(name.split(':')[1:]))


# def merge_bed_info(df, path_to_bed, matching_col):
#     """Read bed and join info on col"""
#     region_bed = pd.read_csv(path_to_bed, sep="\t")
#     region_bed.columns = ['region_' + col_name for col_name in [col_chr, col_start, col_end, col_name, 'score', col_strand]]
#     # merge based on name
#     bed_corrdinates = df.merge(path_to_bed, left_on=col_name, right_on=f'region_{col_name}', how='left')
#     bed_corrdinates

bed_coordinate_beds = {}
pre_metadata_df_filtered.columns
for group_name, bed_path in dnase_control_coordinates.items():
    #
    print(group_name)
    rstrip_shuffled = group_name.split('_shu')[0] if 'shuffled' in group_name else group_name
    region_bed = pd.read_csv(bed_path, sep="\t")
    region_bed.columns = ['region_' + col_name for col_name in [col_chr, col_start, col_end, col_name, 'score', col_strand]]

    # filter metadata_df to only have the bed file with the desired label:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['name'].str.startswith(rstrip_shuffled)].copy()
    # name is suffixed with _. for the not shuffled sequences
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('_.')[0] if '_.' in name else name)
    pre_metadata_df_filtered_group
    print(f'Number of elements in {rstrip_shuffled} group: {pre_metadata_df_filtered_group.shape[0]} ')
    bed_corrdinates = pre_metadata_df_filtered_group.merge(region_bed, left_on='name_no_label_rstrip', right_on='region_name', how='left')
    print(bed_corrdinates.shape[0])
    bed_coordinate_beds[group_name] = bed_corrdinates


GC_DNase_positive
Number of elements in GC_DNase_positive group: 96 
96
GC_DNase_negative_brain
Number of elements in GC_DNase_negative_brain group: 31 
31
GC_DNase_negative_blood
Number of elements in GC_DNase_negative_blood group: 34 
34


### hg19: bed 
- install conda environment liftover:
    - #! conda create -n liftover bioconda::ucsc-liftover
    - #! conda activate liftover
- Use liftover
- liftOver /home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_PosControls.bed.gz hg19ToHg38.over.chain.gz DNase_PosControls_hg38.bed unlifted_DNase_PosControls_hg38.bed
- liftOver /home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.Brain.bed.gz hg19ToHg38.over.chain.gz DNase_NegControls_hg38.bed unlifted_DNase_NegControls_hg38.bed
- liftOver /home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.BloodT-cell.bed.gz hg19ToHg38.over.chain.gz DNase_NegControls.BloodT-cell.bed unlifted_DNase_NegControls.BloodT-cell.bed

#### DNase PosControls
- read new bed file and unlifted file
- for all unlifted: add to info column: split in new; coordinates on hg19
- for all new coordinates: inner join and rename of columns 

In [8]:
bed_coordinate_beds.keys()

dict_keys(['GC_DNase_positive', 'GC_DNase_negative_brain', 'GC_DNase_negative_blood'])

In [9]:
import os

path_to_liftover = '/home/kisa/coding/80K_MPRA/metadata_info/241125_v1_notebooks_data/liftover_dnase_coordinates/DNase_PosControls_hg38.bed'
path_to_unlifted = '/home/kisa/coding/80K_MPRA/metadata_info/241125_v1_notebooks_data/liftover_dnase_coordinates/unlifted_DNase_PosControls_hg38.bed'
output_path = '/'.join(path_to_liftover.split('/')[:-1])
output_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_PosControls'
group_names = ['GC_DNase_positive', 'GC_DNase_positive_shuffeled']
for group_name in group_names:
    print(group_name)
    # if unfiltered: # chr16:46401227-46401496_active_count_113
    unlifted_bed = pd.read_csv(path_to_unlifted, sep="\t", comment="#", header=None)
    unlifted_bed.columns = [ f'unlifted_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]

    # filter metadata_df to only have the bed file with the desired label:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['tmp_label'] == group_name].copy()
    # name is suffixed with _. for the not shuffled sequences
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('_.')[0] if '_.' in name else name)
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('(.)')[0] if '(.)' in name else name)
    # add unlifted information:
    unlifted_regions = pre_metadata_df_filtered_group.merge(unlifted_bed, left_on="name_no_label_rstrip", right_on='unlifted_name', how='inner')
    unlifted_regions[col_ref] = 'GRCh37'
    unlifted_regions[col_category] = 'scrambled'
    unlifted_regions[col_info] = 'Sequences are designed based on GRCh37 (wrong genome build) liftover to GRCh38 not possible sequence Split in new'

    # drop chr, start, end
    unlifted_regions.drop(columns=[col_chr, col_start, col_end], inplace=True)
    prefix = 'unlifted'
    # rename columns
    unlifted_regions.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
    unlifted_regions = unlifted_regions[interesting_columns]
    unlifted_regions.columns
    # # lift over results
    liftover_bed = pd.read_csv(path_to_liftover, sep="\t", comment="#", header=None)
    liftover_bed.columns = [ f'liftover_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]

    lifted_regions = pre_metadata_df_filtered_group.merge(liftover_bed, left_on="name_no_label_rstrip", right_on='liftover_name', how='inner')
    lifted_regions[col_info] = 'Sequences are designed based on GRCh37 (wrong genome build) but liftover'

    lifted_regions[col_category] = 'scrambled' if '_shuff' in group_name else 'element'
    # drop chr, start, end
    lifted_regions.drop(columns=[col_chr, col_start, col_end], inplace=True)

    prefix = 'liftover'
    lifted_regions.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
    lifted_regions = lifted_regions[interesting_columns]

    # check if the row sum is equal to the initial row number
    pre_metadata_df_liftover = pd.concat([lifted_regions, unlifted_regions], ignore_index=True)
    print(f'expected number of rows: {pre_metadata_df_filtered_group.shape[0]}')
    print(f'observed number of rows: {pre_metadata_df_liftover.shape[0]}')
    # Write DataFrame to TSV file
    pre_metadata_df_liftover[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

GC_DNase_positive
expected number of rows: 41
observed number of rows: 41
GC_DNase_positive_shuffeled
expected number of rows: 55
observed number of rows: 55


### Negative controls

#### Blood (+ shuffled)

In [23]:
path_to_liftover = '/home/kisa/coding/80K_MPRA/metadata_info/241125_v1_notebooks_data/liftover_dnase_coordinates/DNase_NegControls.BloodT-cell.bed'
output_path = '/'.join(path_to_liftover.split('/')[:-1])
output_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls'
group_names = ['GC_DNase_negative_blood', 'GC_DNase_negative_blood_shuffeled']
for group_name in group_names:
    print(group_name)
    rstrip_shuffled = group_name.split('_shu')[0] if 'shuffeled' in group_name else group_name

    # filter metadata_df to only have the bed file with the desired label:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['tmp_label'] == group_name].copy()
    # name is suffixed with _. for the not shuffled sequences
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('_.')[0] if '_.' in name else name)
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('(.)')[0] if '(.)' in name else name)

    # # lift over results
    liftover_bed = pd.read_csv(path_to_liftover, sep="\t", comment="#", header=None)
    liftover_bed.columns = [ f'liftover_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]
    lifted_regions = pre_metadata_df_filtered_group.merge(liftover_bed, left_on="name_no_label_rstrip", right_on='liftover_name', how='inner')
    lifted_regions[col_info] = 'Sequences are designed based on GRCh37 (wrong genome build) but liftover'
    lifted_regions[col_category] = 'scrambled' if '_shuff' in group_name else 'element'


    # drop chr, start, end
    lifted_regions.drop(columns=[col_chr, col_start, col_end], inplace=True)

    prefix = 'liftover'
    lifted_regions.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
    lifted_regions = lifted_regions[interesting_columns]

    # check if the row sum is equal to the initial row number
    print(f'expected number of rows: {pre_metadata_df_filtered_group.shape[0]}')
    print(f'observed number of rows: {lifted_regions.shape[0]}')

    # Write DataFrame to TSV file
    lifted_regions[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

GC_DNase_negative_blood
expected number of rows: 15
observed number of rows: 15
GC_DNase_negative_blood_shuffeled
expected number of rows: 19
observed number of rows: 19


In [24]:
path_to_liftover = '/home/kisa/coding/80K_MPRA/metadata_info/241125_v1_notebooks_data/liftover_dnase_coordinates/DNase_NegControls_hg38.bed'
output_path = '/'.join(path_to_liftover.split('/')[:-1])
output_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls'
group_names = ['GC_DNase_negative_brain', 'GC_DNase_negative_brain_shuffeled']
for group_name in group_names:
    print(group_name)
    rstrip_shuffled = group_name.split('_shu')[0] if 'shuffeled' in group_name else group_name

    # filter metadata_df to only have the bed file with the desired label:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['tmp_label'] == group_name].copy()
    # name is suffixed with _. for the not shuffled sequences
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('_.')[0] if '_.' in name else name)
    pre_metadata_df_filtered_group['name_no_label_rstrip'] = pre_metadata_df_filtered_group['name_no_label'].apply(lambda name: name.split('(.)')[0] if '(.)' in name else name)

    # # lift over results
    liftover_bed = pd.read_csv(path_to_liftover, sep="\t", comment="#", header=None)
    liftover_bed.columns = [ f'liftover_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]
    lifted_regions = pre_metadata_df_filtered_group.merge(liftover_bed, left_on="name_no_label_rstrip", right_on='liftover_name', how='inner')
    lifted_regions[col_info] = 'Sequences are designed based on GRCh37 (wrong genome build) but liftover'
    lifted_regions[col_category] = 'scrambled' if '_shuff' in group_name else 'element'

    # drop chr, start, end
    lifted_regions.drop(columns=[col_chr, col_start, col_end], inplace=True)

    prefix = 'liftover'
    lifted_regions.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
    lifted_regions = lifted_regions[interesting_columns]

    # check if the row sum is equal to the initial row number
    print(f'expected number of rows: {pre_metadata_df_filtered_group.shape[0]}')
    print(f'observed number of rows: {lifted_regions.shape[0]}')

    # Write DataFrame to TSV file
    lifted_regions[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

GC_DNase_negative_brain
expected number of rows: 15
observed number of rows: 15
GC_DNase_negative_brain_shuffeled
expected number of rows: 16
observed number of rows: 16


#### Example for 0-based and 1-based h19 corrdinates
- different coordinates in the header compared to the bed file (definition: bed: 0-based)
- GC_DNase_positive_shuffeled: chr1:153959246-153959515_active_count_119
    - fasta sequence: TGCTCAAGGTTTGTACCACCCCTTGAACATGCAGCCCATGGCCTCGTTGGTCTTGGTGCATCCTGAAGGATAAGGGTTAAGTTCAGGACAGGATAACCTCCAATGCTCCACCACCAAATTTGAAAGTTCCCTCTCCTTTTTCC

In [25]:
f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz'

'zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_brain_shuffeled.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_brain_shuffeled.metadata.tsv.gz'

In [24]:
# GC_DNase_positive	/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resourcesd/controls/DNase/DNase_PosControls.bed.gz
# GC_DNase_negative_brain	/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.Brain.bed.gz
# GC_DNase_negative_blood	/home/kisa/coding/80K_MPRA/MPRAOligoDesign/resources/controls/DNase/DNase_NegControls.BloodT-cell.bed.gz
